In [ ]:
import polars as pl

import numpy as np
import os
import itertools
import pandas as pd
from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import PureSVDModel

# Анализ таргета

In [2]:
events_dir = "../data/raw/dataset/small/marketplace/events"

In [3]:
events_df = (
    pl.scan_parquet(f"{events_dir}/*.pq", include_file_paths="path")
    .with_columns(
        pl.col("path")
          .str.extract(r"(\d+)\.pq", group_index=1)
          .cast(pl.Int32)
          .alias("day")
    )
    .sort("day")
    .drop("path")
)

In [4]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [5]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 21h 27m 26s 176761µs,13873047,"""nfmcg_27778856""","""u2i""","""view""","""android""",1082
1082d 21h 27m 26s 706092µs,16678070,"""nfmcg_22186200""","""catalog""","""view""","""android""",1082
1082d 21h 27m 26s 734104µs,72905239,"""nfmcg_4436221""","""u2i""","""view""","""android""",1082
1082d 21h 27m 26s 980200µs,76116652,"""nfmcg_14326289""","""other""","""view""","""android""",1082
1082d 21h 27m 27s 168995µs,66884091,"""nfmcg_11856679""","""u2i""","""view""","""android""",1082


In [6]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""click""",3772094
"""clickout""",485448
"""like""",41792
"""view""",126147783


Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [7]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""click""",34.582149,3.571844,5.880659
"""clickout""",268.714913,5.597366,16.392526
"""like""",3121.341812,8.046339,55.86897
"""view""",1.034082,0.710044,1.016898


In [8]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [9]:
RANDOM_SEED = 42
SUBSET_FRACTION = 0.4

all_user_ids = (
    events_df.select("user_id").unique().collect(engine="streaming")["user_id"].to_list()
)
rng = np.random.default_rng(RANDOM_SEED)
subset_users = set(
    rng.choice(all_user_ids, size=int(len(all_user_ids) * SUBSET_FRACTION), replace=False).tolist()
)
events_df = events_df.filter(pl.col("user_id").is_in(subset_users))

In [10]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 14h 16m 55s 335110µs,80974364,"""nfmcg_12406233""","""u2i""","""view""","""ios""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 14h 16m 55s 427048µs,6202085,"""nfmcg_14092028""","""u2i""","""view""","""ios""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 14h 16m 55s 685338µs,56754067,"""nfmcg_19030396""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 14h 16m 56s 256338µs,25156009,"""nfmcg_21678523""","""other""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 14h 16m 57s 79770µs,45137711,"""nfmcg_12746270""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0


## Global Temporal Split

In [11]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [12]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [13]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [14]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [15]:
train, test = global_temporal_split(events_df, 0.2)

In [16]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).head().collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
84781981,[1285d 8h 14m 41s 169742µs],"[""nfmcg_24504776""]","[""other""]","[""click""]","[""android""]",[1285],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]
89122695,"[1265d 6h 34m 47s 147315µs, 1265d 7h 47m 39s 460638µs, … 1281d 18h 48m 19s 98249µs]","[""nfmcg_18084908"", ""nfmcg_20455166"", … ""nfmcg_7424674""]","[""u2i"", ""u2i"", … ""u2i""]","[""clickout"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1265, 1265, … 1281]","[268.714913, 34.582149, … 34.582149]","[5.597366, 3.571844, … 3.571844]","[16.392526, 5.880659, … 5.880659]","[0, 0, … 0]","[1, 0, … 0]","[0, 0, … 0]","[0, 1, … 1]"
57232378,"[1270d 17h 20m 56s 839959µs, 1279d 13h 16m 34s 245930µs, 1279d 20h 32m 17s 980250µs]","[""nfmcg_25928977"", ""nfmcg_18954296"", ""nfmcg_2501643""]","[""u2i"", ""u2i"", ""u2i""]","[""click"", ""click"", ""click""]","[""android"", ""ios"", ""ios""]","[1270, 1279, 1279]","[34.582149, 34.582149, 34.582149]","[3.571844, 3.571844, 3.571844]","[5.880659, 5.880659, 5.880659]","[0, 0, 0]","[0, 0, 0]","[0, 0, 0]","[1, 1, 1]"
32265994,"[1273d 19h 5m 8s 726761µs, 1274d 14h 22m 45s 879679µs, … 1274d 21h 29m 24s 728520µs]","[""nfmcg_25563242"", ""nfmcg_19023598"", … ""nfmcg_19992403""]","[""other"", ""other"", … ""u2i""]","[""clickout"", ""clickout"", … ""click""]","[""ios"", ""other"", … ""ios""]","[1273, 1274, … 1274]","[268.714913, 268.714913, … 34.582149]","[5.597366, 5.597366, … 3.571844]","[16.392526, 16.392526, … 5.880659]","[0, 0, … 0]","[1, 1, … 0]","[0, 0, … 0]","[0, 0, … 1]"
13020305,[1295d 22h 23m 8s 772190µs],"[""nfmcg_9758746""]","[""u2i""]","[""click""]","[""ios""]",[1295],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]


In [17]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [18]:
from src.dataset import ndcg_at_k

# def ndcg_at_k(
#     user_based_df: pl.DataFrame,
#     relevancy_col: str,
#     true_items_col: str,
#     predicted_items_col: str,
#     predicted_score_col: str,
#     top_k: int = 15,
# ):
#     """
#     Computes user-based NDCG@k for graded relevance in a recommendation setting.

#     Parameters
#     ----------
#     user_based_df : pl.DataFrame
#         Dataframe with user data. Each row must contain user and its lists with: truth
#         ground items, their relevancy estimation and model prediction score.
#     relevancy_col : str
#         Column name contains list of relevancy estimations ground
#         truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
#     true_items_col : str
#         Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
#         of these items must be set in `relevancy_col` respectively. 
#     predicted_items_col : str
#         Columns name with predicted items (pl.List[str]). Must be set in order matches
#         `predicted_score_col`.
#     predicted_score_col : str
#         Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
#         Used to sort predictions in descending order.
#     top_k : int, optional
#         Top k elements to calculate `@k` metric.

#     Returns
#     -------
#     pl.DataFrame
#         Columns:
#         - ``user_id`` : user identifier;
#         - ``ndcg`` : NDCG@k for current user.

#     Notes
#     -----
#     For each user, the function:
#     1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
#     2. Joins predicted items with their ground-truth relevancies.
#     3. Computes DCG@k using the order induced by the model (sorting by score).
#     4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
#     5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
#     """
#     user_ids = []
#     ndcgs = []
#     for row in user_based_df.iter_rows(named=True):
#         true_items = pl.DataFrame(
#             {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
#         )
#         true_items = true_items.group_by("truth_items").agg(
#             pl.col("relevancy").max()
#         )  # Берем максимальную релевантность для товара
#         predictions = (
#             pl.DataFrame(
#                 {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
#             )
#             .join(
#                 true_items,
#                 left_on="predicted_items",
#                 right_on="truth_items",
#                 coalesce=True,
#                 how="left",
#             )
#             .fill_null(0)
#         )
#         idcg = (
#             predictions.select("relevancy")
#             .sort("relevancy", descending=True)
#             .head(top_k)
#             .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
#             .item()
#         )
#         dcg = (
#             predictions.select("score", "relevancy")
#             .sort("score", descending=True)
#             .head(top_k)
#             .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
#             .item()
#         )
#         user_ids.append(row["user_id"])
#         ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
#     return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

2026-06-15 13:09:31.626 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


In [19]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})

In [20]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u2""",0.789998
"""u1""",1.0


## Добавляем метрики Precision@k и Recall@k

Пусть:
- $Rel\_u$ — множество релевантных объектов для пользователя $u$ в тесте.
- $Rec\_u@k$ — множество рекомендованных объектов в топ‑k для пользователя $u$.

Тогда для одного пользователя:


$$
Precision@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{k} = \frac{ hits}{k}
$$
то есть, сколько релевантных из рекомендованных.

$$
Recall@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{|Rel\_u|} = \frac{hits}{Rel\_u}
$$
то есть, сколько релевантных найдено из всех релевантных.

Далее берем среднее по всем пользователям:
$$
Precision@k = \frac{1}{N} \sum_{u=1}^{N} Precision@k(u)
$$
$$
Recall@k = \frac{1}{N} \sum_{u=1}^{N} Recall@k(u)
$$



Precision@k показывает качество самого списка рекомендаций. Она говорит о том, сколько "мусора" мы подсовываем пользователю.

Recall@k показывает охват интересов. Она говорит: "Из всего многообразия товаров, которые этот пользователь мог бы полюбить, какую долю мы реально успели ему показать в нашем коротком топе?".

В рекомендательных системах (особенно когда k маленькое, например 10 или 20) Recall обычно важнее, чем Precision. Precision тоже важна, чтобы не раздражать пользователя нерелевантным мусором, но часто мы готовы пожертвовать точностью (показать пару лишних товаров), лишь бы не упустить то, что пользователь реально ищет (сохранить Recall).

In [21]:
def calculate_metrics(df, k):
    """
    Расчет Precision@k и Recall@k
    df: датафрейм с колонками predicted_items и true_items
    k: количество рекомендаций (TOPK)
    """
    # Обрезаем список предсказаний до k элементов
    top_k_preds = pl.col("predicted_items").list.head(k)
    
    # Ищем пересечение обрезанного списка с правдой
    hits_expr = top_k_preds.list.set_intersection(pl.col("true_items")).list.len()
    
    # Вычисляем метрики одной командой select
    metrics = df.select(
        # Precision = (кол-во попаданий / k), берем среднее по всем юзерам
        (hits_expr / k).mean().alias('precision'),
        
        # Recall = (кол-во попаданий / длину реального списка), берем среднее
        (hits_expr / pl.col('true_items').list.len()).mean().alias('recall')
    )
    
    precision_val = metrics['precision'].item()
    recall_val = metrics['recall'].item()

    return precision_val, recall_val

## Добавляем метрики RMSE и MAE

In [22]:
def calculate_regression_metrics_vectorized(
    test_data,
    user_features: np.ndarray,
    item_features: np.ndarray,
    user_to_idx: dict,
    item_to_idx: dict,
    target_col: str = "log_target",
) -> dict:
    
    # Фильтруем только известные пары user-item
    test_filtered = test_data.filter(
        pl.col("user_id").is_in(list(user_to_idx.keys())) &
        pl.col("item_id").is_in(list(item_to_idx.keys()))
    )
    
    # Получаем индексы
    user_ids = test_filtered["user_id"].to_list()
    item_ids = test_filtered["item_id"].to_list()
    true_scores = test_filtered[target_col].to_numpy()
    
    user_idxs = np.array([user_to_idx[uid] for uid in user_ids])
    item_idxs = np.array([item_to_idx[iid] for iid in item_ids])
    
    # Вычисление скоров
    # pred_score[i] = user_features[user_idx[i]] @ item_features[:, item_idx[i]]
    pred_scores = np.sum(
        user_features[user_idxs] * item_features[:, item_idxs].T, 
        axis=1
    )
    
    # Метрики
    errors = true_scores - pred_scores
    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))
    
    return {
        "rmse": rmse,
        "mae": mae,
        "n_samples": len(true_scores),
    }

Комментарий к коду выше:

* Результат user_predictions — это вектор, где в каждой ячейке лежит число, показывающее "силу связи" (значимость) между этим пользователем и конкретным товаром. Чем больше число, тем сильнее мы хотим порекомендовать этот товар.

* ind_sorted это массив индексов товаров. Индексы идут в порядке убывания скора товара. То есть ind_sorted[0] — это индекс самого подходящего товара, ind_sorted[1] — второго по крутости и т.д.

* top_k_item_ids это оригинальные ID товаров

* top_k_scores это те значимости (скоры), которые мы посчитали (user_predictions). Мы достаем их из большого массива user_predictions по нужным индексам.

Замечание: в функции выше мы исключаем товары, с которыми пользователь взаимодействовал. Скорее всего это не совсем правильный подход. Может быть, стоит исключать только те, которые были просмотренные, а остальные оставлять. 

# Будем реализовывать Pure SVD

https://rectools.readthedocs.io/en/stable/api/rectools.models.pure_svd.PureSVDModel.html

>PureSVD matrix factorization model. See https://dl.acm.org/doi/10.1145/1864708.1864721

Источник: https://rectools.readthedocs.io/en/stable/api/rectools.models.pure_svd.PureSVDModel.html

В PureSVDModelD используются приближённые алгоритмы SVD. Ищутся k наибольших сингулярных значений и соответствующие им сингулярные векторы, а вклад всех остальных считается пренебрежимо малым и отбрасывается.

`PureSVDModel` минимизирует норму Фробениуса разности между исходной матрицей и её приближением.

# Описание бейзлайна

**Будем реализовывать Pure SVD c количеством латентных факторов LATENT_FACTORS = 30 и потом выбирать TOP_K = 15 рекомендаций**

1) Из `train` выбираются столбцы `user_id`, `item_id`, `target` / `sqrt_target` / `log_target` и из них составляется DataFrame `train_data`
2) Фильтруем датафрейм так, чтобы остались только пользователи и айтемы с количеством взаимодейтсвий >= 5
3) Переводим оригинальные id юзеров и айтемов в простые числовые индексы
4) Строим датасет в rectools совместимом формате. Мокаем признак даты-времени (для rectools он обязателен, но SVD его не использует).

5) Для восстановления тех значений, которые были нулями будем использовать `PureSVD` из `rectools`.

>`PureSVD` (сингулярное разложение) берет нашу разреженную матрицу взаимодействий (где строки — пользователи, столбцы —    товары) и приближенно раскладывает её в произведение трех матриц:
>
>$$ R \approx U \times \Sigma \times V^T $$
>
>Здесь:
>1. $U$ - показывает отношение пользователей к скрытым факторам.
>2. $\Sigma$ - диагональная матрица с сингулярными числами. Эти числа показывают важность каждого скрытого фактора. Чем больше число, тем больше этот фактор влияет на итоговый выбор.
>3. $V^T$ - показывает, насколько каждый скрытый фактор выражен в конкретном товаре.
>
>

In [ ]:
def run_svd_experiment(train_data: pl.DataFrame, test_data, target_col='log_target', latent_factors=20, top_k=15):
    print(f'🚀 Запуск эксперимента: Target={target_col}, Factors={latent_factors}, TopK={top_k}')

    train_data = train_data.select(['user_id', 'item_id', target_col, "timestamp"]).collect(engine='streaming')

    MIN_INTERACTIONS = 5
    user_counts = train_data.group_by('user_id').agg(pl.len().alias('user_count'))
    item_counts = train_data.group_by('item_id').agg(pl.len().alias('item_count'))
    valid_users = user_counts.filter(pl.col('user_count') >= MIN_INTERACTIONS).select('user_id')
    valid_items = item_counts.filter(pl.col('item_count') >= MIN_INTERACTIONS).select('item_id')

    train_data = (
        train_data
        .join(valid_users, on='user_id', how="inner")
        .join(valid_items, on='item_id', how="inner")
    )
    print(f'Взаимодействий после фильтрации: {train_data.height}')

    # ── 1. Dataset ──────────────────────────────────────────────────────────
    interactions_pd = (
        train_data
        .select(["user_id", "item_id", target_col, "timestamp"])
        .rename({"user_id": Columns.User, "item_id": Columns.Item, target_col: Columns.Weight, "timestamp": Columns.Datetime})
        .to_pandas()
    )
    interactions_pd[Columns.Datetime] = pd.Timestamp("2020-01-01") # мокаем дату-время, для SVD это неважно
    dataset = Dataset.construct(interactions_df=interactions_pd)

    # ── 2. Обучение ──────────────────────────────────────────────────────────
    model = PureSVDModel(factors=latent_factors, recommend_n_threads=16)
    model.fit(dataset)

    # ── 3. Подготовка тестовых данных ────────────────────────────────────────
    test_data_collected = test_data.select(['user_id', 'item_id', target_col]).collect(engine='streaming')

    known_users = set(train_data['user_id'].to_list())
    test_data_filtered = test_data_collected.filter(pl.col('user_id').is_in(known_users))
    print(f"Пользователей в тесте после фильтрации: {test_data_filtered.select('user_id').n_unique()}")

    user_test_truth = (
        test_data_filtered.group_by('user_id')
        .agg(
            pl.col('item_id').alias('true_items'),
            pl.col(target_col).alias('relevancy'),
        )
    )

    # ── 4. Генерация рекомендаций ─────────────────────────────────────────────
    test_users_pd = pd.DataFrame({Columns.User: user_test_truth['user_id'].to_list()})

    reco = model.recommend(
        users=user_test_truth['user_id'].to_list(),
        dataset=dataset,
        k=top_k,
        filter_viewed=True,
    )
    # reco — pandas DataFrame: [user_id, item_id, score, rank]

    # Конвертируем в формат списков, который ожидают функции оценки
    reco_grouped = (
        pl.from_pandas(reco)
        .sort([Columns.User, Columns.Rank])
        .group_by(Columns.User, maintain_order=True)
        .agg([
            pl.col(Columns.Item).alias('predicted_items'),
            pl.col(Columns.Score).alias('predicted_scores'),
        ])
        .rename({Columns.User: 'user_id'})
    )

    evaluation_df = user_test_truth.join(reco_grouped, on='user_id', how='inner')

    # ── 5. Ранжирующие метрики ────────────────────────────────────────────────
    ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col='relevancy',
        true_items_col='true_items',
        predicted_items_col='predicted_items',
        predicted_score_col='predicted_scores',
        top_k=top_k,
    )
    mean_ndcg = ndcg_results['ndcg'].mean()
    precision, recall = calculate_metrics(evaluation_df, k=top_k)

    # ── 6. Метрики регрессии ──────────────────────────────────────────────────
    user_factors, item_factors = model.get_vectors()

    # Восстанавливаем маппинг external_id -> internal_idx через IdMap
    user_ext_ids = dataset.user_id_map.get_external_sorted_by_internal()
    item_ext_ids = dataset.item_id_map.get_external_sorted_by_internal()
    user_to_idx = {uid: idx for idx, uid in enumerate(user_ext_ids)}
    item_to_idx = {iid: idx for idx, iid in enumerate(item_ext_ids)}

    metrics = calculate_regression_metrics_vectorized(
        test_data_filtered,
        user_features=user_factors,          # (n_users, factors)
        item_features=item_factors.T,        # (factors, n_items)
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        target_col=target_col,
    )

    return {
        'target': target_col,
        'latent_factors': latent_factors,
        'top_k': top_k,
        'ndcg': mean_ndcg,
        'precision': precision,
        'recall': recall,
        'rmse': metrics['rmse'],
        'mae': metrics['mae'],
    }

In [25]:
train.head().collect()

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 1s 59212µs,33049410,"""nfmcg_801936""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 1s 552075µs,82199233,"""nfmcg_27615847""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0
1082d 1s 870870µs,14844486,"""nfmcg_24900487""","""u2i""","""view""","""android""",1082,1.034082,0.710044,1.016898,1,0,0,0


In [26]:
# Запуск всех экспериментов
experiment_configs = [
    # Базовые сравнения таргетов
    ('log_target', 20, 15),
    ('sqrt_target', 20, 15),
    ('target', 20, 15),
    
    # Дополнительные проверки для log_target 
    ('log_target', 50, 20),
    ('log_target', 30, 15) 
]

results = []

for t_col, factors, k in experiment_configs:
    print(f"Запускаю: {t_col}, factors={factors}, k={k}...")

    res = run_svd_experiment(train, test, target_col=t_col, latent_factors=factors, top_k=k)
    results.append(res)

final_df = pl.DataFrame(results)
print(final_df)

Запускаю: log_target, factors=20, k=15...
🚀 Запуск эксперимента: Target=log_target, Factors=20, TopK=15
Взаимодействий после фильтрации: 38907655
Пользователей в тесте после фильтрации: 254446
Запускаю: sqrt_target, factors=20, k=15...
🚀 Запуск эксперимента: Target=sqrt_target, Factors=20, TopK=15
Взаимодействий после фильтрации: 38907655
Пользователей в тесте после фильтрации: 254446
Запускаю: target, factors=20, k=15...
🚀 Запуск эксперимента: Target=target, Factors=20, TopK=15
Взаимодействий после фильтрации: 38907655
Пользователей в тесте после фильтрации: 254446
Запускаю: log_target, factors=50, k=20...
🚀 Запуск эксперимента: Target=log_target, Factors=50, TopK=20
Взаимодействий после фильтрации: 38907655
Пользователей в тесте после фильтрации: 254446
Запускаю: log_target, factors=30, k=15...
🚀 Запуск эксперимента: Target=log_target, Factors=30, TopK=15
Взаимодействий после фильтрации: 38907655
Пользователей в тесте после фильтрации: 254446
shape: (5, 8)
┌─────────────┬────────────

## Выводы

В результате сравнения разных таргетов при LATENT_FACTORS = 20 и TOP_K = 15 наилучшие значения ранжирующих метрик (NDCG, Precision@k, Recall@k) получены при использовании log_target по сравнению с sqrt_target и сырым target (см. таблицу ниже).

Лучший вариант среди всех проведённых экспериментов — конфигурация с log_target, LATENT_FACTORS = 20 и TOP_K = 15: она даёт максимально высокий NDCG и наилучший баланс Precision@k и Recall@k.

При увеличении числа латентных факторов до 30 NDCG падает до 0.031, Precision и Recall также ухудшаются. Это указывает на то, что усложнение модели не приводит к улучшению качества рекомендаций и, вероятно, начинает подхватывать больше шума.

Конфигурация log_target, LATENT_FACTORS = 50 и TOP_K = 20 даёт ещё более низкие значения NDCG и Precision@k. Это ожидаемо: рост числа факторов повышает риск переобучения, а увеличение k приводит к тому, что в топ‑список попадает больше нерелевантных объектов.

Метрики регрессии для лог‑таргета показывают, что SVD лишь грубо аппроксимирует численные значения log_target. Это ожидаемо: PureSVD оптимизирует приближение всей матрицы взаимодействий и не нацелен на точное восстановление таргета. При LATENT_FACTORS = 50 и TOP_K = 20 RMSE (3.17) и MAE (1.01) становятся немного выше, при этом ранжирующие метрики снижаются, что подтверждает, что улучшение по RMSE/MAE не связано с улучшением качества рекомендаций.